In [1]:
import time
import json
import requests
import re

import pandas as pd
pd.set_option('display.max_columns', None)

from tqdm.notebook import tqdm_notebook

PRODUCT_URL = "https://world.openfoodfacts.org/api/v2/product"

In [2]:
df_mups = pd.read_excel('MUP.csv')
df_mups.head()

,category,mup_id,mup_name,individual_compound,match_type,is_extended
0,cosmetic_additives__flavours,1,flavour,flavor,exact,False
1,cosmetic_additives__flavours,1,flavour,flavour,exact,False
2,cosmetic_additives__flavour_enhancers,2,e62*,e620 - e629,prefix,False
3,cosmetic_additives__flavour_enhancers,3,e63*,e630 - e637,prefix,False
4,cosmetic_additives__flavour_enhancers,4,e640,e640,exact,False


## Section 1: Inspect Scraped Data

### **1.1 Inspect Waitrose Data** 

In [3]:
df_waitrose = pd.read_json('/files/problem-set-1-grayusan/data/scraped/products.jsonl', lines=True)
df_waitrose.head()

,name,barcode,url,food_type,pricing
0,Norfolk County French Vanilla Dairy Ice Cream,[5017809000013],https://www.waitrose.com/ecom/products/norfolk...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",8.0
1,Waitrose Cinnamon Bun Dairy Ice Cream,[5063210019939],https://www.waitrose.com/ecom/products/waitros...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",3.5
2,Waitrose Frozen 2 Smoked Haddock Fillets MSC,[5000169578995],https://www.waitrose.com/ecom/products/waitros...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",7.5
3,Remeo Madagascan Vanilla Gelato,[5060392490020],https://www.waitrose.com/ecom/products/remeo-m...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",6.1
4,Waitrose Frozen 2 Haddock Fillets MSC,[5000169578988],https://www.waitrose.com/ecom/products/waitros...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",6.5


### **1.2 OpenFoodFacts responses**

In [9]:
openfoodfacts_responses = []

sample = df_waitrose['barcode'].head(5)

for idx, barcodes in tqdm_notebook(sample.items()):
    row_results = []
    for barcode in barcodes:
        response = requests.get(f'{PRODUCT_URL}/{barcode}', timeout=30)
        if response.status_code == 200:
            result = response.json()
            row_results.append({
                'product_id': idx,
                'barcode_used': barcode,
                'response': result,
            })
        time.sleep(0.5)

    if not row_results:
        openfoodfacts_responses.append({
            'product_id': idx,
            'barcode_used': barcodes[0] if barcodes else None,
            'response': None,
        })

    else:
        openfoodfacts_responses.extend(row_results)

with open("/files/problem-set-1-grayusan/api/off_data_validation_sample.json", "w") as f:
    json.dump(openfoodfacts_responses, f, indent=2, ensure_ascii=False)
print(f"Saved {len(openfoodfacts_responses)} responses to off_data_validation_sample.json")

0it [00:00, ?it/s]

Saved 5 responses to off_data_validation_sample.json


In [4]:
with open('off_data_validation_sample.json', 'r') as f:
    api_data = json.load(f)

In [6]:
df_api = pd.json_normalize(api_data, meta=['response', 'product'])
df_api.columns.to_list()

['product_id',
 'barcode_used',
 'response',
 'response.code',
 'response.product._id',
 'response.product._keywords',
 'response.product.added_countries_tags',
 'response.product.allergens',
 'response.product.allergens_from_ingredients',
 'response.product.allergens_from_user',
 'response.product.allergens_hierarchy',
 'response.product.allergens_lc',
 'response.product.allergens_tags',
 'response.product.brands',
 'response.product.brands_tags',
 'response.product.categories',
 'response.product.categories_hierarchy',
 'response.product.categories_lc',
 'response.product.categories_properties.agribalyse_proxy_food_code:en',
 'response.product.categories_properties.ciqual_food_code:en',
 'response.product.categories_properties_tags',
 'response.product.categories_tags',
 'response.product.checkers_tags',
 'response.product.cities_tags',
 'response.product.code',
 'response.product.codes_tags',
 'response.product.compared_to_category',
 'response.product.complete',
 'response.product.